# Fraud Detection: Proof of Concept
### Using Machine Learning to Identify Suspicious Credit Card Transactions

**Prepared by:** Ayumi Tanaka

**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), 284,807 transactions

---

## What This Document Is

This is an interactive notebook, a document that combines plain-language explanations with the actual code that runs our fraud detection model. **You do not need to understand the code.** Each code section is followed by a clear explanation of what it does and why it matters to the business.

Think of this as a guided walkthrough of the fraud detection process, written for decision-makers rather than programmers.

---

## The Business Problem

Our existing fraud detection system is generating too many **false positives** — legitimate customer transactions are being incorrectly flagged as suspicious. This creates two problems:

1. **Customer experience:** Valid purchases are being blocked or delayed, frustrating customers.
2. **Operational cost:** Fraud analysts spend time reviewing cases that turn out to be legitimate.

At the same time, some real fraud is slipping through undetected, resulting in direct financial losses.

This notebook tests a machine learning approach that learns fraud patterns from historical data, with the goal of improving detection accuracy over time.

---

## Pipeline Overview

The process follows seven steps:

```
┌──────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  Step 1          │────▶│  Step 2          │────▶│  Step 3          │
│  Import Tools    │     │  Load Dataset    │     │  Prepare Data    │
└──────────────────┘     └──────────────────┘     └──────────────────┘
                                                           │
                                                           ▼
┌──────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  Step 6          │◀────│  Step 5          │◀────│  Step 4          │
│  Register Model  │     │  Evaluate Model  │     │  Train Model     │
└──────────────────┘     └──────────────────┘     └──────────────────┘
        │
        ▼
┌──────────────────────┐
│  Step 7              │
│  Visualize Results   │
└──────────────────────┘
```

---

## Azure ML Components Used

This notebook runs on Microsoft Azure Machine Learning Studio — a secure, cloud-based platform for building and managing AI models. The table below explains the role of each component.

| Azure ML Component | Role in This Project |
|---|---|
| **Workspace** | The central hub that organizes all project resources, experiments, and outputs |
| **Dataset (Datastore)** | Secure cloud storage for the 284,807 credit card transactions used to train the model |
| **Compute Cluster** | Cloud-based computing power that runs the model training job |
| **Experiment** | A named container that tracks each model training run and its results |
| **Environment** | Defines the software libraries needed to run the code consistently |
| **Model Registry** | Stores the trained model so it can be reused, versioned, or deployed later |
| **SHAP Explainer** | A tool that explains which data features most influenced the model's decisions |

---

## About the Dataset

The dataset contains **284,807 real credit card transactions** made over two days by European cardholders. Of these:

- **284,315** are legitimate transactions
- **492** are confirmed fraud cases — just **0.17%** of the total

This extreme imbalance is one of the biggest challenges in fraud detection. A model that simply labels every transaction as "legitimate" would appear to be 99.8% accurate — but it would catch zero fraud. This is why we need more nuanced measures of success, which are explained in Step 5.

---

## Workflow

### Step 1: Import Tools

**What this does:** Before any analysis can begin, the notebook loads the software tools it needs — similar to opening the right applications before starting a task.

**Why it matters:** Each tool has a specific role:
- **Azure ML tools** connect this notebook to the secure cloud environment where the data is stored and the model will be saved.
- **Pandas** is a data analysis tool that lets us work with the transaction dataset as a structured table, like a very powerful spreadsheet.
- **Isolation Forest** is the machine learning algorithm that will detect suspicious transactions.
- **Classification Report** generates the performance summary showing how well the model worked.

In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

### Step 2: Load the Credit Card Fraud Dataset

**What this does:** This step connects to our Azure cloud environment and retrieves the credit card transaction dataset that was previously uploaded and registered there. It then loads the data into a table format that the model can work with.

**Why it matters:** Rather than storing sensitive financial data on individual laptops, Azure provides a secure, centralized location. This ensures that everyone working on the project is using the same version of the data, and that access is controlled and auditable.

> ⚠️ **Note for reviewers:** The dataset is approximately 150MB in size. When loading for the first time, this step may take up to 4 minutes. Any warning messages that appear during this step are normal system messages and can be safely ignored. They do not indicate errors.

> 💡 **Running on Kaggle instead of Azure:** This notebook was also adapted to run directly on Kaggle without an Azure account, making it easier to experiment with the model without cloud infrastructure setup. The adapted version is available in this repository.

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

### Step 3: Prepare the Data

**What this does:** Before the model can learn from the data, the transaction amounts need to be rescaled so they are comparable to the other 28 data columns. The `Time` column is also removed, as it does not meaningfully help identify fraud in this context. Finally, the data is split into two parts: the features used to make predictions (`X`), and the known fraud labels used to measure accuracy (`y`).

**Why it matters:** Machine learning models are sensitive to the scale of numbers they receive. Without rescaling, a raw transaction amount of $10,000 would dominate the model's decisions compared to a value of 0.5 in another column — even if that other column is a much stronger fraud signal. This step ensures the model evaluates all features fairly and on equal footing.

> 📊 **Data quality note:** Our prior analysis of this dataset found that the `Amount` column has an extreme skewness value of 16.98, one of the most distorted columns in the dataset. Rescaling helps reduce this distortion, but log-transformation would be an even more effective improvement for a future version of this model.

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Step 4: Train the Model

**What this does:** This step trains the fraud detection model using an algorithm called **Isolation Forest**. The model studies the full dataset to learn what a "normal" transaction looks like, then learns to flag anything that deviates significantly from that pattern.

**Why it matters:** Unlike a traditional rule-based system (for example, "flag any transaction over $5,000"), this model learns patterns automatically from real data. It does not need to be told in advance what fraud looks like. It identifies transactions that are statistically unusual compared to the vast majority of legitimate ones. This makes it more flexible and adaptable to new fraud patterns over time.

**How Isolation Forest works in plain English:**

Imagine sorting a pile of cards by randomly dividing them into smaller and smaller groups. Normal cards, which look similar to each other — take many splits to separate out. Unusual cards, which stand out from the rest — get isolated very quickly with just a few splits. The model uses this same principle: **transactions that are easy to isolate are flagged as potential fraud.**

**Key setting:** The `contamination` parameter (set to 0.0017) tells the model to expect approximately 0.17% of transactions to be suspicious — matching the known fraud rate in our dataset. This helps calibrate how aggressively the model flags anomalies.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 5: Evaluate the Model

**What this does:** This step compares the model's predictions with the known fraud labels in the dataset to see how well it performed.

**Why overall accuracy is misleading here:** Because fraud is extremely rare, a model could look “good” simply by calling almost everything legitimate. That would not be useful in practice. What matters more is whether the model finds real fraud without causing too many unnecessary alerts.

**Understanding the performance metrics:**

| Metric | Plain-English Meaning |
|---|---|
| **Precision** | Of all the transactions the model flagged as fraud, how many were actually fraud? |
| **Recall** | Of all the real fraud cases in the dataset, how many did the model successfully catch? |
| **F1-Score** | A combined score that balances precision and recall — higher is better |
| **Support** | The total number of actual cases in each category |

**Our actual results:**

| | Predicted Legitimate | Predicted Fraud |
|---|---|---|
| **Actually Legitimate** | 283,965 ✅ | 350 ❌ |
| **Actually Fraud** | 357 ❌ | 135 ✅ |

**What this means in business terms:**
- The model correctly caught **135 out of 492 fraud cases** — a **27% fraud catch rate**.
- It missed **357 real fraud transactions** that went undetected.
- It incorrectly blocked **350 legitimate transactions**, causing unnecessary friction for customers.

> ⚠️ **Business implication:** In its current form, this model is a proof of concept, not a production-ready system. The 27% fraud catch rate tells us the model needs further refinement before deployment. The good news is that we now have a clear baseline and understand exactly where to improve. See the Business Impact Assessment at the end of this notebook.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 6 (Optional): Register the Model

**What this does:** Once we are satisfied with the model's performance, this step saves it to the Azure ML Model Registry — a secure, versioned storage system for trained models.

**Why it matters:** Registering the model means it can be:
- **Deployed** into a live production system to start evaluating real transactions
- **Versioned** so that future improvements can be tracked and compared against this baseline
- **Shared** with other teams or systems without re-running the full training pipeline

> 💡 This step is marked optional because it is only needed when we are ready to move beyond the proof-of-concept stage. Our current focus is on understanding and improving the model's performance before committing to a deployment.

In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### Step 7: Visualize the Results

#### Chart 1: Count of Predicted Anomalies

**What this shows:** A bar chart comparing how many transactions the model classified as normal versus suspicious.

**How to read it:**
- The bar on the left (`0`) shows transactions the model considered normal — this bar will be very tall, since the vast majority of transactions are legitimate.
- The bar on the right (`1`) shows transactions the model flagged as potential fraud — this bar should be very short, ideally close to the actual number of fraud cases (492).

**What to look for:** If the fraud bar is much taller than 492, the model is being too aggressive and generating too many false alarms. If it is much shorter, the model is too cautious and missing real fraud. A well-calibrated model will flag a number close to the actual fraud count.

<img src="./img/chart1.png" width="400" height="300">

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


#### Chart 2: Transaction Amount by Prediction Class

**What this shows:** A box plot comparing the dollar amounts of transactions the model predicted as normal versus suspicious.

**How to read it:**
- Each box represents the range of transaction amounts for that prediction category.
- The line in the middle of each box is the median (middle value) transaction amount.
- Dots outside the boxes are outliers — transactions with unusually high or low amounts.

**What to look for:** If the fraud box (`1`) shows a wider spread or more extreme values than the legitimate box (`0`), it suggests the model is partially identifying fraud by flagging unusual transaction amounts. This is a reasonable signal, but amount alone is not sufficient to reliably detect fraud — which is why the model uses 29 features in total.

<img src="./img/chart2.png" width="400" height="300">

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


#### Chart 3: SHAP Feature Importance — What Is the Model Actually Looking At?

**What this shows:** This chart called a SHAP beeswarm plot reveals which data features most influenced the model's decisions, and in which direction.

**How to read it:**
- Each row represents one data feature (e.g. `V1`, `V14`, `Amount`).
- Each dot represents one transaction.
- **Colour** shows the value of that feature: red = high value, blue = low value.
- **Horizontal position** shows the impact on the model's prediction: dots to the right push the model toward flagging fraud; dots to the left push toward predicting normal.
- Features at the **top of the chart** had the most influence on the model's decisions overall.

**Why this matters for stakeholders:** SHAP charts are one of the most important tools for building trust in AI models. They answer the question: *"Why did the model flag this transaction?"* Without this kind of transparency, it is very difficult to audit, defend, or improve a model's decisions — especially in a regulated industry like financial services.

> 📌 **Note:** Only the first 100 transactions are used here to keep the chart fast and readable. The full dataset would show the same patterns at a larger scale.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

---

## Business Impact Assessment

This section summarizes the business implications of our proof-of-concept model results for non-technical stakeholders. It covers four key areas: the cost of model errors, recommendations for improvement, risks and how to mitigate them, and how to communicate model limitations to each audience.

---

### 1. Cost-Benefit Analysis: False Positives vs. Missed Fraud

Our model produces two types of errors, each with a distinct business cost:

| Error Type | Definition | Count in Our Test | Business Cost |
|---|---|---|---|
| **False Positive** | A legitimate transaction incorrectly flagged as fraud | 350 | Customer frustration, blocked purchases, wasted analyst time, potential churn |
| **False Negative** | A real fraud transaction that went undetected | 357 | Direct financial loss, customer harm, regulatory exposure, reputational risk |

**Which error is more costly?** In financial services, missed fraud (false negatives) typically carries the higher financial cost — each undetected fraudulent transaction represents a direct monetary loss. However, false positives carry a significant indirect cost: customers whose valid purchases are blocked are likely to lose trust in the service, and in a competitive market, this can directly impact retention.

**The current trade-off:** Our model catches 135 out of 492 fraud cases (27%) while incorrectly blocking 350 legitimate transactions. At this performance level, the cost of missed fraud likely outweighs the benefit of what the model catches. Before deployment, the model must be improved to a point where the value of fraud prevented meaningfully exceeds the combined cost of missed fraud and customer disruption.

**What good looks like:** A production-ready fraud model for this dataset would ideally achieve a fraud recall (catch rate) of 80% or higher, with a false positive rate low enough that fewer than 1 in 1,000 legitimate transactions are incorrectly flagged.

---

### 2. Recommendations for Model Improvement and Deployment

The following improvements were identified through analysis of the dataset and model results, with assistance from Claude (Anthropic's AI assistant).

**Data quality improvements (highest priority):**

- **Log-transform the `Amount` field** — With a skewness of 16.98, this is the most distorted column in the dataset and is likely contributing to poor model performance. A log-transformation would bring it closer to a normal distribution before training.
- **Winsorize high-outlier columns** — `V27` (13.75% outlier rate), `V28` (10.65%), and `V23` (6.51%) all contain extreme values that distort the model's sense of "normal." Capping these at the 1st and 99th percentile before retraining is a low-effort, high-impact improvement.
- **Address class imbalance with SMOTE** — With only 492 fraud cases out of 284,807 transactions, the model has very few fraud examples to learn from. SMOTE (Synthetic Minority Over-sampling Technique) generates realistic synthetic fraud cases, giving the model a more balanced training signal.

**Algorithm improvements:**

- **Switch to a supervised model** — Because we have labeled fraud data, a supervised classifier such as XGBoost or Random Forest can learn directly from confirmed fraud examples, rather than trying to infer fraud from statistical rarity alone. Supervised models consistently outperform Isolation Forest on labeled datasets of this type.
- **Ensemble approach** — Combine the Isolation Forest output with a supervised model score to produce a more robust prediction than either model alone.

**Deployment recommendations:**

- **Do not deploy the current model to production.** The 27% fraud catch rate is insufficient for a live environment.
- **Retrain after data improvements** and re-evaluate results before making a deployment decision.
- **Implement a human review queue** for borderline predictions rather than making fully automated block or allow decisions.
- **Set up monthly performance monitoring** to track fraud catch rate and false positive rate after deployment, and trigger retraining when either metric degrades.

---

### 3. Risk Assessment and Mitigation Strategies

| Risk | Likelihood | Potential Impact | Mitigation Strategy |
|---|---|---|---|
| Model deployed before it is ready | Medium | High — significant missed fraud and customer disruption | Require sign-off from the fraud team and compliance before any production deployment |
| Fraud patterns evolve and model becomes stale | High | Medium — declining catch rate over time | Schedule quarterly model retraining on fresh transaction data |
| Unintended bias against certain transaction types or regions | Low | High — regulatory and legal exposure | Legal and compliance review of model outputs before deployment; ongoing bias monitoring |
| False positives damage customer trust at scale | Medium | Medium — customer churn and service complaints | Implement a human review queue for borderline cases; set a maximum false positive rate threshold before going live |
| Data privacy breach | Low | High — regulatory penalties and reputational damage | All transaction data remains anonymized; Azure security controls enforce access restrictions |
| Over-reliance on model without human oversight | Medium | High — missed fraud that analysts would have caught | Model should augment, not replace, fraud analysts in the near term |

---

### 4. Stakeholder Communication Plan for Model Limitations

Transparency about what the model can and cannot do is essential for maintaining trust and enabling informed decisions. The following communication approach is recommended for each audience.

**Executive leadership:**
> This proof of concept demonstrates that a machine learning approach to fraud detection is technically feasible using our existing transaction data. However, the current model catches only 27% of fraud cases and is not yet suitable for production. We have identified specific, actionable improvements that are expected to significantly increase that rate. We are requesting approval to proceed with a second iteration before a deployment decision is made.

**Fraud analysis team:**
> The model provides a ranked list of transactions most likely to be fraudulent. In its current form, it is best used as a triage tool — a first filter that helps prioritize which transactions to review — rather than an automated decision-maker. Human judgment remains essential, particularly for borderline cases the model is uncertain about.

**Customer service team:**
> During the testing and improvement phase, a small number of legitimate customer transactions may be incorrectly flagged. This is a known limitation of the current model and is being actively addressed. If customers report unexpected transaction declines, this should be treated as a priority escalation path for the fraud team.

**Legal and compliance:**
> Before any production deployment, the model's outputs will be reviewed for unintended bias — for example, whether certain transaction amounts, merchant categories, or geographic patterns are being disproportionately flagged. A full compliance review is a mandatory step in the deployment checklist, not an afterthought.

**General principle:** The model is a tool to support human decision-making, not to replace it. All final fraud decisions in the near term should remain with trained analysts. As the model's performance improves and trust is established, the level of automation can be progressively increased.

---

## Use of AI in This Project

Claude (Anthropic's AI assistant) was used in the following ways during this project:

- **Documentation:** The plain-language explanations throughout this notebook were drafted with Claude's assistance and reviewed to ensure accuracy and relevance.

All interpretations, recommendations, and business conclusions in this notebook reflect the author's own understanding and judgment.